In [13]:
import sys
import numpy as np

sys.path.append("../../../")
from Rain import Rain

sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

In [14]:
import sys
sys.path.append('../../../')
from clean_all import clean
clean()
sys.path.pop()

'../../../'

In [15]:
config = {
    "lib": "tensorflow",
    "partitions": 3,
    "iterations": 3,
    "lr": 0.001,
    "epochs": 2,
    "batch_size": 128,
    "loss": tf.keras.losses.CategoricalCrossentropy(),
    "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),
}

In [16]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )


def partition_train_data(X_train, y_train, partitions):
    num_samples = X_train.shape[0]

    # Create an array of indices from 0 to num_samples - 1
    indices = np.arange(num_samples)

    # Shuffle the indices
    np.random.shuffle(indices)

    # Use the shuffled indices to shuffle the datasets
    X_train = X_train[indices]
    y_train = y_train[indices]

    X_train_partitions = []
    y_train_partitions = []

    partition_size = int(len(X_train) / partitions)

    for i in range(partitions):
        if i == partitions - 1:
            X_train_partitions.append(X_train[i * partition_size :])
            y_train_partitions.append(y_train[i * partition_size :])
        else:
            X_train_partitions.append(
                X_train[i * partition_size : (i + 1) * partition_size]
            )
            y_train_partitions.append(
                y_train[i * partition_size : (i + 1) * partition_size]
            )

    return X_train_partitions, y_train_partitions

In [17]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [18]:
X_train, y_train = get_train_data()
X_train, y_train = partition_train_data(X_train, y_train, config["partitions"])

In [19]:
for i in range(len(X_train)):
    np.save(f"../../../data/X_train_{i + 1}.npy", X_train[i])
    np.save(f"../../../data/y_train_{i + 1}.npy", y_train[i])

In [20]:
model = create_model()
rain = Rain(config, model, X_train, y_train)

Rain is initialized
Provisioner created successfully
divider is running
Deleting network security group: Rain_nic-nsg


Exception ignored in: <function Provisioner.__del__ at 0x7f654ed80280>
Traceback (most recent call last):
  File "/media/mostafa/CUFE/GP/rain/Examples/MNIST_model/tensorflow/../../../Provisioner/Provisioner.py", line 77, in __del__
    raise Exception("Error deleting networking")
Exception: Error deleting networking


(InvalidAuthenticationTokenTenant) The access token is from the wrong issuer 'https://sts.windows.net/77255288-5298-4ea5-81aa-a13e604c30ac/'. It must match the tenant 'https://sts.windows.net/818e33cd-1e2a-461b-922c-edb76eb87d31/' associated with this subscription. Please use the authority (URL) 'https://login.windows.net/818e33cd-1e2a-461b-922c-edb76eb87d31' to get the token. Note, if the subscription is transferred to another tenant there is no impact to the services, but information about new tenant could take time to propagate (up to an hour). If you just transferred your subscription and see this error message, please try back later.
Code: InvalidAuthenticationTokenTenant
Message: The access token is from the wrong issuer 'https://sts.windows.net/77255288-5298-4ea5-81aa-a13e604c30ac/'. It must match the tenant 'https://sts.windows.net/818e33cd-1e2a-461b-922c-edb76eb87d31/' associated with this subscription. Please use the authority (URL) 'https://login.windows.net/818e33cd-1e2a-46

In [21]:
# rain.setup_vms()

In [22]:
# rain.delete_vms()

In [25]:
model = rain.train_centralized_sync()

divider received: Success receiving the number of workers from provisioner
divider is sending data to the coordinator


In [24]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 1s 7ms/step - loss: 0.0971 - accuracy: 0.9705

Test accuracy: 97.0%
